# Sensitivity analysis: large municipalities

Recompute the coverage-population slope, Gini coefficient and Moran's I after restricting each data source to municipalities that are large relative to the approximate Level 13 Bing tile size used in the Facebook data.

This notebook uses source-specific samples: the multi-app and Facebook metrics are computed on each source's own available municipalities, rather than on the intersection of both sources.


In [28]:
from pathlib import Path

import geopandas as gpd
import numpy as np
import pandas as pd
from esda.moran import Moran
from libpysal.weights import Queen
from scipy.optimize import curve_fit
from scipy import stats


## Parameters

The restricted sample uses municipalities with area at least 5 times the approximate Level 13 Bing tile area in Mexico. The Level 13 tile area varies by latitude; here, 20 km2 is used as the approximate Mexico-specific reference value.


In [29]:
ROOT = Path.cwd().parents[2] if Path.cwd().name == "code" else Path("/Users/carmen/Documents/github/de-bias/DEBIAS_MX")

FB_PATH = ROOT / "results/measure_bias/facebook_tts/Active_population_bias_nonspatial.csv"
MP_PATH = ROOT / "results/measure_bias/phone_00_08/Active_population_bias_nonspatial.csv"
MUNICIPALITIES_PATH = ROOT / "data/raw/spatial/municipalities_measure_bias/MunicipiosMX.shp"

OUTPUT_PATH = ROOT / "manuscript/figures/data/sensitivity_large_municipalities_metrics.csv"

TILE_AREA_KM2 = 20.0
THRESHOLD_MULTIPLIER = 5
AREA_THRESHOLD_KM2 = TILE_AREA_KM2 * THRESHOLD_MULTIPLIER

AREA_THRESHOLD_KM2


100.0

## Helper functions


In [30]:
def power_func(x, a, b):
    return a * x**b


def coverage_population_slope(data, coverage_col):
    xdata = data["population"].to_numpy(dtype=float)
    ydata = data[coverage_col].to_numpy(dtype=float)
    popt, _ = curve_fit(power_func, xdata, ydata, p0=[1, 1], maxfev=10000)
    return popt[0], popt[1]


def lorenz_model(x, k, p):
    return (1 - k) * x**p + k * (1 - (1 - x) ** (1 / p))


def fitted_lorenz_gini(data, active_col):
    ordered = data.sort_values("population").reset_index(drop=True)
    share_pop = ordered["population"] / ordered["population"].sum()
    share_active = ordered[active_col] / ordered[active_col].sum()

    # Follows the Figure 1 notebook convention.
    cum_pop = np.array([share_pop.iloc[:i].sum() for i in range(len(ordered))], dtype=float)
    cum_active = np.array([share_active.iloc[:i].sum() for i in range(len(ordered))], dtype=float)

    fit_order = np.argsort(cum_pop)
    xdata = cum_pop[fit_order]
    ydata = cum_active[fit_order]

    popt, _ = curve_fit(
        lorenz_model,
        xdata,
        ydata,
        p0=[0.2, 1.5],
        bounds=([0.0, 0.1], [1.0, 5.0]),
        maxfev=10000,
    )
    _, p_hat = popt
    return (p_hat - 1) / (p_hat + 1)


def empirical_gini(values):
    x = np.asarray(values, dtype=float)
    x = x[~np.isnan(x)]
    if len(x) == 0 or np.sum(x) == 0:
        return np.nan
    if np.min(x) < 0:
        x = x - np.min(x)
    x = np.sort(x)
    n = len(x)
    index = np.arange(1, n + 1)
    return (2 * np.sum(index * x) / (n * np.sum(x))) - ((n + 1) / n)


def morans_i(areas, data, bias_col):
    gdf = areas.merge(data[["Area_code", bias_col]], on="Area_code", how="inner")
    gdf = gdf.dropna(subset=[bias_col]).reset_index(drop=True)

    weights = Queen.from_dataframe(gdf, use_index=False)
    weights.transform = "r"

    result = Moran(gdf[bias_col].to_numpy(dtype=float), weights, permutations=999)
    return result.I, result.p_sim, len(gdf)


def moran_plot_slope(areas, data, bias_col):
    gdf = areas.merge(data[["Area_code", bias_col]], on="Area_code", how="inner")
    gdf = gdf.dropna(subset=[bias_col]).reset_index(drop=True)

    weights = Queen.from_dataframe(gdf, use_index=False)
    weights.transform = "r"
    lag = weights.sparse @ gdf[bias_col].to_numpy(dtype=float)

    xdata = gdf[bias_col].to_numpy(dtype=float)
    ydata = np.asarray(lag, dtype=float)
    popt, pcov = curve_fit(lambda x, a, b: a + x * b, xdata, ydata, p0=[1, 1], maxfev=10000)
    a_hat, b_hat = popt
    perr = np.sqrt(np.diag(pcov))
    dof = max(0, len(xdata) - len(popt))
    t_b = b_hat / perr[1]
    p_b = 2 * (1 - stats.t.cdf(np.abs(t_b), dof))
    return b_hat, p_b


## Load municipality areas


In [31]:
areas = gpd.read_file(MUNICIPALITIES_PATH)
areas = areas.rename(columns={areas.columns[0]: "Area_code"})
areas["Area_code"] = areas["Area_code"].astype(int)

try:
    areas_area = areas[["Area_code", "geometry"]].to_crs("EPSG:6372")
except Exception:
    areas_area = areas[["Area_code", "geometry"]].to_crs("EPSG:8857")

areas_area["area_km2"] = areas_area.geometry.area / 1_000_000
areas_area = areas_area[["Area_code", "area_km2"]]

areas_area.head()


,Area_code,area_km2
0,1001,1166.359248
1,1002,543.374266
2,1003,923.357832
3,1004,128.383545
4,1005,499.895092


## Load source-specific analysis samples

These samples intentionally remain source-specific. This avoids changing the multi-app sample simply because Facebook has missing municipalities.


In [32]:
fb = pd.read_csv(FB_PATH, encoding="latin1")
mp = pd.read_csv(MP_PATH, encoding="latin1")

fb_df = fb[["Area_code", "Area_name", "Total_Population", "Total_active_population"]].rename(
    columns={"Total_Population": "population", "Total_active_population": "active"}
)
fb_df["data_source"] = "Facebook"

mp_df = mp[["Area_code", "Area_name", "Total_Population", "Total_active_population"]].rename(
    columns={"Total_Population": "population", "Total_active_population": "active"}
)
mp_df["data_source"] = "Multi-app"

source_data = {
    "Multi-app": mp_df,
    "Facebook": fb_df,
}

for source, data in source_data.items():
    data = data.merge(areas_area, on="Area_code", how="left")
    data = data.dropna(subset=["population", "active", "area_km2"]).reset_index(drop=True)
    data["coverage"] = data["active"] / data["population"]
    data = data[data["coverage"] <= 1].reset_index(drop=True)
    data["bias"] = 1 - data["coverage"]
    source_data[source] = data
    print(source, "full sample n =", len(data))


Multi-app full sample n = 2461
Facebook full sample n = 2077


## Define source-specific restricted samples


In [33]:
restricted_data = {}

for source, data in source_data.items():
    restricted = data[data["area_km2"] >= AREA_THRESHOLD_KM2].copy().reset_index(drop=True)
    restricted_data[source] = restricted
    print(source, "restricted sample n =", len(restricted))

print("Area threshold km2:", AREA_THRESHOLD_KM2)


Multi-app restricted sample n = 1765
Facebook restricted sample n = 1747
Area threshold km2: 100.0


## Compute metrics


In [34]:
def summarise_source_sample(areas, data, sample_name, source, area_threshold_km2=None):
    _, beta = coverage_population_slope(data, "coverage")
    moran_i, moran_p, moran_n = morans_i(areas, data, "bias")
    slope, slope_p = moran_plot_slope(areas, data, "bias")

    return {
        "sample": sample_name,
        "data_source": source,
        "n_municipalities": len(data),
        "n_moran_municipalities": moran_n,
        "area_threshold_km2": area_threshold_km2,
        "coverage_population_beta": beta,
        "fitted_lorenz_gini": fitted_lorenz_gini(data, "active"),
        "empirical_coverage_gini": empirical_gini(data["coverage"]),
        "morans_i": moran_i,
        "morans_i_p_value": moran_p,
        "moran_plot_slope": slope,
        "moran_plot_slope_p_value": slope_p,
    }


rows = []
for source, data in source_data.items():
    rows.append(summarise_source_sample(areas, data, "Full source-specific sample", source))
    rows.append(
        summarise_source_sample(
            areas,
            restricted_data[source],
            f"Municipalities >= {AREA_THRESHOLD_KM2:.0f} km2",
            source,
            AREA_THRESHOLD_KM2,
        )
    )

sensitivity_metrics = pd.DataFrame(rows)
sensitivity_metrics


/Users/carmen/anaconda3/envs/geo-env-new/lib/python3.10/site-packages/libpysal/weights/weights.py:224: UserWarning: The weights matrix is not fully connected: 
 There are 2 disconnected components.
  warnings.warn(message)
/Users/carmen/anaconda3/envs/geo-env-new/lib/python3.10/site-packages/libpysal/weights/weights.py:224: UserWarning: The weights matrix is not fully connected: 
 There are 2 disconnected components.
  warnings.warn(message)
/Users/carmen/anaconda3/envs/geo-env-new/lib/python3.10/site-packages/libpysal/weights/weights.py:224: UserWarning: The weights matrix is not fully connected: 
 There are 3 disconnected components.
 There is 1 island with id: 1532.
  warnings.warn(message)


('WARNING: ', 1532, ' is an island (no neighbors)')


/Users/carmen/anaconda3/envs/geo-env-new/lib/python3.10/site-packages/libpysal/weights/weights.py:224: UserWarning: The weights matrix is not fully connected: 
 There are 3 disconnected components.
 There is 1 island with id: 1532.
  warnings.warn(message)


('WARNING: ', 1532, ' is an island (no neighbors)')


/Users/carmen/anaconda3/envs/geo-env-new/lib/python3.10/site-packages/libpysal/weights/weights.py:224: UserWarning: The weights matrix is not fully connected: 
 There are 4 disconnected components.
 There are 3 islands with ids: 269, 1038, 1728.
  warnings.warn(message)


('WARNING: ', 269, ' is an island (no neighbors)')
('WARNING: ', 1038, ' is an island (no neighbors)')
('WARNING: ', 1728, ' is an island (no neighbors)')


/Users/carmen/anaconda3/envs/geo-env-new/lib/python3.10/site-packages/libpysal/weights/weights.py:224: UserWarning: The weights matrix is not fully connected: 
 There are 4 disconnected components.
 There are 3 islands with ids: 269, 1038, 1728.
  warnings.warn(message)


('WARNING: ', 269, ' is an island (no neighbors)')
('WARNING: ', 1038, ' is an island (no neighbors)')
('WARNING: ', 1728, ' is an island (no neighbors)')


,sample,data_source,n_municipalities,n_moran_municipalities,area_threshold_km2,coverage_population_beta,fitted_lorenz_gini,empirical_coverage_gini,morans_i,morans_i_p_value,moran_plot_slope,moran_plot_slope_p_value
0,Full source-specific sample,Multi-app,2461,2461,NaN,0.296852,0.256451,0.439020,0.545602,0.001,0.545602,0.000000
1,Municipalities >= 100 km2,Multi-app,1765,1765,100.0,0.285115,0.270691,0.406206,0.498032,0.001,0.589701,0.000509
2,Full source-specific sample,Facebook,2077,2077,NaN,0.027347,0.120894,0.494439,0.295144,0.001,0.321531,0.000000
3,Municipalities >= 100 km2,Facebook,1747,1747,100.0,0.051971,0.122539,0.473642,0.270982,0.001,0.270982,0.000000


## Save results


In [ ]:
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
sensitivity_metrics.to_csv(OUTPUT_PATH, index=False)
OUTPUT_PATH
